In [1]:
from pathlib import Path
from pynq import Overlay, MMIO
import time

WORK_DIR = Path.cwd()
BIT_PATH = WORK_DIR / "design_1.bit"
HWH_PATH = WORK_DIR / "design_1.hwh"

print(f"Notebook directory: {WORK_DIR}")
print(f"Bitstream path: {BIT_PATH}")
print(f"Hardware description path: {HWH_PATH}")

assert BIT_PATH.is_file(), f"Missing file: {BIT_PATH}"
assert HWH_PATH.is_file(), f"Missing file: {HWH_PATH}"

overlay = Overlay(str(BIT_PATH), download=True)

print("Overlay loaded successfully")

Notebook directory: /home/xilinx/jupyter_notebooks/overlay_pynq
Bitstream path: /home/xilinx/jupyter_notebooks/overlay_pynq/design_1.bit
Hardware description path: /home/xilinx/jupyter_notebooks/overlay_pynq/design_1.hwh


Overlay loaded successfully


In [2]:
def to_int(value):
    if isinstance(value, str):
        return int(value, 0)

    return int(value)


ip_entries = dict(overlay.ip_dict)
mem_entries = dict(getattr(overlay, "mem_dict", {}))

print("IP dictionary entries:")

for name, info in ip_entries.items():
    print(f"{name}: {info.get('type', 'unknown')}")

print("")
print("Memory dictionary entries:")

for name, info in mem_entries.items():
    print(f"{name}: {info.get('type', 'unknown')}")


addressable_entries = {}

for source in (ip_entries, mem_entries):
    for name, info in source.items():
        if "phys_addr" not in info:
            continue

        if "addr_range" not in info:
            continue

        addressable_entries[name] = info


print("")
print("Addressable entries:")

for name, info in addressable_entries.items():
    base_address = to_int(info["phys_addr"])
    address_range = to_int(info["addr_range"])

    print(
        f"{name}: "
        f"base=0x{base_address:08X}, "
        f"range=0x{address_range:X}, "
        f"type={info.get('type', 'unknown')}"
    )


gpio_names = []

bram_names = []

for name, info in addressable_entries.items():
    description = " ".join(
        [
            name,
            str(info.get("type", "")),
            str(info.get("fullpath", "")),
        ]
    ).lower()

    if "axi_gpio" in description:
        gpio_names.append(name)

    if "axi_bram_ctrl" in description:
        bram_names.append(name)


gpio_names = sorted(set(gpio_names))
bram_names = sorted(set(bram_names))

print("")
print(f"GPIO candidates: {gpio_names}")
print(f"BRAM candidates: {bram_names}")

assert len(gpio_names) == 1, (
    f"Expected exactly one AXI GPIO, found {gpio_names}"
)

assert len(bram_names) == 2, (
    f"Expected exactly two AXI BRAM controllers, found {bram_names}"
)

GPIO_NAME = gpio_names[0]
BRAM_NAMES = bram_names

print("")
print(f"Selected GPIO: {GPIO_NAME}")
print(f"Detected BRAM controllers: {BRAM_NAMES}")

IP dictionary entries:
axi_gpio_0: xilinx.com:ip:axi_gpio:2.0
processing_system7_0: xilinx.com:ip:processing_system7:5.5

Memory dictionary entries:
axi_bram_ctrl_0: DDR4
axi_bram_ctrl_1: DDR4
PSDDR: DDR4

Addressable entries:
axi_gpio_0: base=0x41200000, range=0x10000, type=xilinx.com:ip:axi_gpio:2.0
axi_bram_ctrl_0: base=0x40000000, range=0x1000, type=DDR4
axi_bram_ctrl_1: base=0x42000000, range=0x1000, type=DDR4

GPIO candidates: ['axi_gpio_0']
BRAM candidates: ['axi_bram_ctrl_0', 'axi_bram_ctrl_1']

Selected GPIO: axi_gpio_0
Detected BRAM controllers: ['axi_bram_ctrl_0', 'axi_bram_ctrl_1']


In [3]:
def get_address_info(name):
    if name in overlay.ip_dict:
        return overlay.ip_dict[name]

    memory_dict = getattr(overlay, "mem_dict", {})

    if name in memory_dict:
        return memory_dict[name]

    raise KeyError(f"Address information not found for: {name}")


def create_mmio(name):
    info = get_address_info(name)

    base_address = to_int(info["phys_addr"])
    address_range = to_int(info["addr_range"])

    assert address_range >= 0x44, (
        f"Address range is too small for {name}: "
        f"0x{address_range:X}"
    )

    print(
        f"Creating MMIO for {name}: "
        f"base=0x{base_address:08X}, "
        f"range=0x{address_range:X}"
    )

    return MMIO(base_address, address_range)


gpio = create_mmio(GPIO_NAME)

bram_mmio = {
    name: create_mmio(name)
    for name in BRAM_NAMES
}


GPIO_DATA = 0x00
GPIO_TRI = 0x04

RUN_RESET_ASSERTED = 0x00000000
RUN_RESET_RELEASED = 0x00000001

RESULT_OFFSET = 0x40
SAFE_LOOP = 0x0000006F

PROGRAM = [
    0x00500093,
    0x00700113,
    0x002081B3,
    0x04302023,
    0x0000006F,
]


def write_word(mmio, byte_offset, value):
    mmio.write(
        int(byte_offset),
        int(value) & 0xFFFFFFFF,
    )


def read_word(mmio, byte_offset):
    return int(
        mmio.read(int(byte_offset))
    ) & 0xFFFFFFFF


def hold_cpu_in_reset():
    gpio.write(GPIO_TRI, 0x00000000)
    gpio.write(GPIO_DATA, RUN_RESET_ASSERTED)
    time.sleep(0.01)


def run_cpu_test(imem_name, dmem_name):
    imem = bram_mmio[imem_name]
    dmem = bram_mmio[dmem_name]

    hold_cpu_in_reset()

    for bram_name in BRAM_NAMES:
        memory = bram_mmio[bram_name]

        write_word(memory, 0x00, SAFE_LOOP)
        write_word(memory, RESULT_OFFSET, 0x00000000)

    for index, instruction in enumerate(PROGRAM):
        write_word(
            imem,
            index * 4,
            instruction,
        )

    readback = [
        read_word(imem, index * 4)
        for index in range(len(PROGRAM))
    ]

    if readback != PROGRAM:
        expected_text = [
            f"0x{value:08X}"
            for value in PROGRAM
        ]

        actual_text = [
            f"0x{value:08X}"
            for value in readback
        ]

        raise RuntimeError(
            "Instruction memory readback failed. "
            f"Expected {expected_text}, "
            f"got {actual_text}"
        )

    try:
        gpio.write(
            GPIO_DATA,
            RUN_RESET_RELEASED,
        )

        time.sleep(0.05)

        result = read_word(
            dmem,
            RESULT_OFFSET,
        )
    finally:
        gpio.write(
            GPIO_DATA,
            RUN_RESET_ASSERTED,
        )

        time.sleep(0.01)

    return result, readback


hold_cpu_in_reset()

print("MMIO objects created")
print("CPU is held in reset")
print("Test functions are ready")

Creating MMIO for axi_gpio_0: base=0x41200000, range=0x10000
Creating MMIO for axi_bram_ctrl_0: base=0x40000000, range=0x1000
Creating MMIO for axi_bram_ctrl_1: base=0x42000000, range=0x1000
MMIO objects created
CPU is held in reset
Test functions are ready


In [4]:
selected_imem = None
selected_dmem = None
final_result = None

mapping_candidates = [
    (BRAM_NAMES[0], BRAM_NAMES[1]),
    (BRAM_NAMES[1], BRAM_NAMES[0]),
]

for imem_name, dmem_name in mapping_candidates:
    print("")
    print(f"Testing IMEM={imem_name}, DMEM={dmem_name}")

    result, readback = run_cpu_test(
        imem_name=imem_name,
        dmem_name=dmem_name,
    )

    print("Instruction memory readback:")

    for index, value in enumerate(readback):
        print(f"IMEM[{index}] = 0x{value:08X}")

    print(
        f"DMEM[0x{RESULT_OFFSET:02X}] = "
        f"0x{result:08X}"
    )

    if result == 12:
        selected_imem = imem_name
        selected_dmem = dmem_name
        final_result = result
        break

hold_cpu_in_reset()

assert final_result == 12, (
    "Hardware CPU test failed for both BRAM mappings"
)

print("")
print("Hardware CPU test passed")
print(f"Selected IMEM: {selected_imem}")
print(f"Selected DMEM: {selected_dmem}")
print(f"Result: {final_result}")
print("CPU is held in reset")


Testing IMEM=axi_bram_ctrl_0, DMEM=axi_bram_ctrl_1
Instruction memory readback:
IMEM[0] = 0x00500093
IMEM[1] = 0x00700113
IMEM[2] = 0x002081B3
IMEM[3] = 0x04302023
IMEM[4] = 0x0000006F
DMEM[0x40] = 0x0000000C

Hardware CPU test passed
Selected IMEM: axi_bram_ctrl_0
Selected DMEM: axi_bram_ctrl_1
Result: 12
CPU is held in reset


In [5]:
assert final_result == 12, "Cell 4 did not pass"

IMEM_NAME = selected_imem
DMEM_NAME = selected_dmem

imem = bram_mmio[IMEM_NAME]
dmem = bram_mmio[DMEM_NAME]

imem_info = get_address_info(IMEM_NAME)
dmem_info = get_address_info(DMEM_NAME)

print(f"IMEM: {IMEM_NAME}")
print(f"IMEM base: 0x{to_int(imem_info['phys_addr']):08X}")

print(f"DMEM: {DMEM_NAME}")
print(f"DMEM base: 0x{to_int(dmem_info['phys_addr']):08X}")

print("Hardware mapping saved")

IMEM: axi_bram_ctrl_0
IMEM base: 0x40000000
DMEM: axi_bram_ctrl_1
DMEM base: 0x42000000
Hardware mapping saved


In [6]:
result = read_word(dmem, RESULT_OFFSET)

print(f"DMEM address: 0x{RESULT_OFFSET:08X}")
print(f"DMEM value: 0x{result:08X}")
print(f"DMEM decimal value: {result}")

assert result == 12, (
    f"Expected 12, got {result}"
)

print("Final hardware verification passed")

DMEM address: 0x00000040
DMEM value: 0x0000000C
DMEM decimal value: 12
Final hardware verification passed


In [7]:
def load_program(program_words, imem_clear_bytes=256):
    assert len(program_words) > 0, "Program is empty"
    assert imem_clear_bytes % 4 == 0, (
        "IMEM clear size must be word-aligned"
    )
    assert len(program_words) * 4 <= imem_clear_bytes, (
        "Program is larger than the cleared IMEM region"
    )

    hold_cpu_in_reset()

    for offset in range(0, imem_clear_bytes, 4):
        write_word(imem, offset, SAFE_LOOP)

    for index, instruction in enumerate(program_words):
        write_word(imem, index * 4, instruction)

    readback = [
        read_word(imem, index * 4)
        for index in range(len(program_words))
    ]

    expected = [
        int(instruction) & 0xFFFFFFFF
        for instruction in program_words
    ]

    assert readback == expected, (
        "Program readback verification failed"
    )

    print(f"Loaded {len(program_words)} instructions")
    print("Program readback verified")


def clear_data_memory(start_offset=0, size_bytes=256):
    assert start_offset % 4 == 0, (
        "DMEM start offset must be word-aligned"
    )
    assert size_bytes % 4 == 0, (
        "DMEM size must be word-aligned"
    )

    hold_cpu_in_reset()

    for offset in range(
        start_offset,
        start_offset + size_bytes,
        4,
    ):
        write_word(dmem, offset, 0)

    print(
        f"Cleared {size_bytes} bytes of data memory "
        f"from 0x{start_offset:08X}"
    )


def start_cpu(run_time_seconds=0.05):
    assert run_time_seconds > 0, (
        "Run time must be positive"
    )

    gpio.write(GPIO_DATA, RUN_RESET_RELEASED)
    time.sleep(run_time_seconds)
    gpio.write(GPIO_DATA, RUN_RESET_ASSERTED)
    time.sleep(0.01)

    print(
        f"CPU ran for {run_time_seconds:.3f} seconds "
        "and returned to reset"
    )


def dump_data_memory(start_offset=0, word_count=16):
    assert start_offset % 4 == 0, (
        "DMEM start offset must be word-aligned"
    )
    assert word_count > 0, (
        "Word count must be positive"
    )

    values = []

    for index in range(word_count):
        offset = start_offset + index * 4
        value = read_word(dmem, offset)
        values.append(value)

        print(
            f"DMEM[0x{offset:08X}] = "
            f"0x{value:08X} ({value})"
        )

    return values


hold_cpu_in_reset()

print("Reusable hardware control functions are ready")

Reusable hardware control functions are ready


In [8]:
TEST_PROGRAM = [
    0x00500093,
    0x00700113,
    0x002081B3,
    0x04302023,
    0x0000006F,
]

clear_data_memory(
    start_offset=0,
    size_bytes=256,
)

load_program(
    program_words=TEST_PROGRAM,
    imem_clear_bytes=256,
)

start_cpu(
    run_time_seconds=0.05,
)

result = read_word(
    dmem,
    RESULT_OFFSET,
)

print(
    f"Result at DMEM[0x{RESULT_OFFSET:08X}] = "
    f"0x{result:08X} ({result})"
)

assert result == 12, (
    f"Expected 12, got {result}"
)

print("Reusable program loader test passed")

Cleared 256 bytes of data memory from 0x00000000
Loaded 5 instructions
Program readback verified
CPU ran for 0.050 seconds and returned to reset
Result at DMEM[0x00000040] = 0x0000000C (12)
Reusable program loader test passed


In [9]:
memory_dump = dump_data_memory(
    start_offset=0,
    word_count=20,
)

DMEM[0x00000000] = 0x00000000 (0)
DMEM[0x00000004] = 0x00000000 (0)
DMEM[0x00000008] = 0x00000000 (0)
DMEM[0x0000000C] = 0x00000000 (0)
DMEM[0x00000010] = 0x00000000 (0)
DMEM[0x00000014] = 0x00000000 (0)
DMEM[0x00000018] = 0x00000000 (0)
DMEM[0x0000001C] = 0x00000000 (0)
DMEM[0x00000020] = 0x00000000 (0)
DMEM[0x00000024] = 0x00000000 (0)
DMEM[0x00000028] = 0x00000000 (0)
DMEM[0x0000002C] = 0x00000000 (0)
DMEM[0x00000030] = 0x00000000 (0)
DMEM[0x00000034] = 0x00000000 (0)
DMEM[0x00000038] = 0x00000000 (0)
DMEM[0x0000003C] = 0x00000000 (0)
DMEM[0x00000040] = 0x0000000C (12)
DMEM[0x00000044] = 0x00000000 (0)
DMEM[0x00000048] = 0x00000000 (0)
DMEM[0x0000004C] = 0x00000000 (0)


In [10]:
FULL_TEST_RESULT_OFFSET = 0x40
FULL_TEST_EXPECTED = 0x12345048

FULL_TEST_PROGRAM = [
    0x123450B7,
    0x00500113,
    0x00700193,
    0x00310233,
    0x00402023,
    0x00002283,
    0x00428333,
    0x01800393,
    0x00730463,
    0x04002023,
    0x006084B3,
    0x0080046F,
    0x04002023,
    0x008484B3,
    0x04902023,
    0x0000006F,
]

hold_cpu_in_reset()

clear_data_memory(
    start_offset=0,
    size_bytes=256,
)

load_program(
    program_words=FULL_TEST_PROGRAM,
    imem_clear_bytes=256,
)

start_cpu(
    run_time_seconds=0.05,
)

full_test_result = read_word(
    dmem,
    FULL_TEST_RESULT_OFFSET,
)

print(
    f"Full test result: "
    f"0x{full_test_result:08X}"
)

print(
    f"Expected result: "
    f"0x{FULL_TEST_EXPECTED:08X}"
)

assert full_test_result == FULL_TEST_EXPECTED, (
    f"Full hardware test failed: "
    f"expected 0x{FULL_TEST_EXPECTED:08X}, "
    f"got 0x{full_test_result:08X}"
)

print("Full hardware regression test passed")

Cleared 256 bytes of data memory from 0x00000000
Loaded 16 instructions
Program readback verified
CPU ran for 0.050 seconds and returned to reset
Full test result: 0x12345048
Expected result: 0x12345048
Full hardware regression test passed


In [11]:
required_names = [
    "overlay",
    "gpio",
    "imem",
    "dmem",
    "hold_cpu_in_reset",
    "clear_data_memory",
    "load_program",
    "start_cpu",
    "read_word",
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

assert not missing_names, (
    f"Run the initialization cells first: {missing_names}"
)

print("All prerequisites are ready")

All prerequisites are ready


In [12]:
SLT_TEST_PROGRAM = [
    0xFFF00093,
    0x00100113,
    0x0020A1B3,
    0x00112233,
    0x02302023,
    0x02402223,
    0x0000006F,
]

hold_cpu_in_reset()

clear_data_memory(
    start_offset=0x20,
    size_bytes=8,
)

load_program(
    program_words=SLT_TEST_PROGRAM,
    imem_clear_bytes=256,
)

start_cpu(
    run_time_seconds=0.01,
)

slt_negative_less_positive = read_word(dmem, 0x20)
slt_positive_less_negative = read_word(dmem, 0x24)

print(
    "SLT(-1, 1): "
    f"{slt_negative_less_positive}"
)

print(
    "SLT(1, -1): "
    f"{slt_positive_less_negative}"
)

assert slt_negative_less_positive == 1, (
    "Signed SLT test failed for -1 < 1"
)

assert slt_positive_less_negative == 0, (
    "Signed SLT test failed for 1 < -1"
)

print("Signed SLT hardware test passed")

Cleared 8 bytes of data memory from 0x00000020
Loaded 7 instructions
Program readback verified
CPU ran for 0.010 seconds and returned to reset
SLT(-1, 1): 1
SLT(1, -1): 0
Signed SLT hardware test passed


In [13]:
from pathlib import Path
import random
import time

ARRAY_BASE = 0x00
ARRAY_LENGTH = 32
ARRAY_END = ARRAY_BASE + ARRAY_LENGTH * 4

DONE_OFFSET = ARRAY_END
RIGHT_GUARD_OFFSET = DONE_OFFSET + 4

DONE_MAGIC = 0xCAFEBABE
RIGHT_GUARD_VALUE = 0x2468ACE0

RANDOM_SEED = 20260806
rng = random.Random(RANDOM_SEED)

negative_values = [
    rng.randint(-0x80000000, -1)
    for _ in range(16)
]

positive_values = [
    rng.randint(1, 0x7FFFFFFF)
    for _ in range(16)
]

SORT_INPUT = negative_values + positive_values
rng.shuffle(SORT_INPUT)

assert len(SORT_INPUT) == ARRAY_LENGTH

assert any(
    value < 0
    for value in SORT_INPUT
), "Input must contain negative integers"

assert any(
    value > 0
    for value in SORT_INPUT
), "Input must contain positive integers"

assert all(
    -0x80000000 <= value <= 0x7FFFFFFF
    for value in SORT_INPUT
), "Input contains a value outside signed 32-bit range"


SORT_PROGRAM = [
    0x00000093,
    0x01F00113,
    0x00010193,
    0x00008213,
    0x00022283,
    0x00422303,
    0x005323B3,
    0x00038663,
    0x00622023,
    0x00522223,
    0x00420213,
    0xFFF18193,
    0x00018463,
    0xFDDFF06F,
    0xFFF10113,
    0x00010463,
    0xFC9FF06F,
    0xCAFEC437,
    0xABE40413,
    0x08802023,
    0x0000006F,
]


SORT_ASSEMBLY = """\
.section .text
.globl _start

# Register usage:
# x1: base address of the 32-element array
# x2: outer-loop counter
# x3: inner-loop counter
# x4: address of the current array element
# x5: current array value
# x6: next array value
# x7: signed comparison result
# x8: completion magic value

_start:
    addi x1, x0, 0
    addi x2, x0, 31

outer_loop:
    addi x3, x2, 0
    addi x4, x1, 0

inner_loop:
    lw   x5, 0(x4)
    lw   x6, 4(x4)
    slt  x7, x6, x5
    beq  x7, x0, no_swap

    sw   x6, 0(x4)
    sw   x5, 4(x4)

no_swap:
    addi x4, x4, 4
    addi x3, x3, -1
    beq  x3, x0, end_inner
    jal  x0, inner_loop

end_inner:
    addi x2, x2, -1
    beq  x2, x0, sorting_done
    jal  x0, outer_loop

sorting_done:
    # Construct 0xCAFEBABE in x8.
    lui  x8, 0xCAFEC
    addi x8, x8, -1346

    # Write the completion magic value to DMEM address 0x80.
    sw   x8, 128(x0)

halt:
    jal  x0, halt
"""


def unsigned_word(value):
    return int(value) & 0xFFFFFFFF


def signed_word(value):
    value = unsigned_word(value)

    if value & 0x80000000:
        return value - 0x100000000

    return value


def save_hex_file(file_path, words):
    text = "\n".join(
        f"{unsigned_word(word):08X}"
        for word in words
    )

    Path(file_path).write_text(
        text + "\n",
        encoding="ascii",
    )


def load_hex_words(file_path):
    path = Path(file_path)

    assert path.is_file(), (
        f"Missing Hex file: {path}"
    )

    words = []

    for line_number, line in enumerate(
        path.read_text(
            encoding="ascii"
        ).splitlines(),
        start=1,
    ):
        text = line.strip()

        if not text:
            continue

        try:
            word = int(text, 16)
        except ValueError as error:
            raise ValueError(
                f"Invalid Hex word at line "
                f"{line_number}: {text}"
            ) from error

        assert 0 <= word <= 0xFFFFFFFF

        words.append(word)

    return words


Path("test_sort.s").write_text(
    SORT_ASSEMBLY,
    encoding="ascii",
)

print(f"Random seed: {RANDOM_SEED}")
print(f"Array start: 0x{ARRAY_BASE:02X}")
print(f"Array end: 0x{ARRAY_END - 4:02X}")
print(f"Done address: 0x{DONE_OFFSET:02X}")
print(f"Instruction count: {len(SORT_PROGRAM)}")
print("Random signed input:")
print(SORT_INPUT)

Random seed: 20260806
Array start: 0x00
Array end: 0x7C
Done address: 0x80
Instruction count: 21
Random signed input:
[-186330981, 249585318, 894641537, -624010408, -1003773380, -1960796514, 249913527, -578235842, 608433994, 100786595, 1633279752, 677325013, -362282644, 1741598169, -728255358, -330939495, 1560925635, 1550625964, -2039872380, -534524188, 587641507, 21737741, 1716811657, 470093155, -975196147, -1426982517, 1152064071, -1989937412, -948628915, -159491399, 1872679488, -1429315569]


In [14]:
hold_cpu_in_reset()

TEST_SORT_HEX_WORDS = (
    SORT_PROGRAM
    + [SAFE_LOOP] * (
        64 - len(SORT_PROGRAM)
    )
)

save_hex_file(
    "test_sort.hex",
    TEST_SORT_HEX_WORDS,
)

program_from_hex = load_hex_words(
    "test_sort.hex"
)

assert program_from_hex[:len(SORT_PROGRAM)] == SORT_PROGRAM, (
    "Machine-code Hex readback failed"
)

clear_data_memory(
    start_offset=0,
    size_bytes=256,
)

load_program(
    program_words=program_from_hex,
    imem_clear_bytes=256,
)

write_word(
    dmem,
    DONE_OFFSET,
    0,
)

write_word(
    dmem,
    RIGHT_GUARD_OFFSET,
    RIGHT_GUARD_VALUE,
)

for index, value in enumerate(SORT_INPUT):
    write_word(
        dmem,
        ARRAY_BASE + index * 4,
        unsigned_word(value),
    )


input_readback = [
    signed_word(
        read_word(
            dmem,
            ARRAY_BASE + index * 4,
        )
    )
    for index in range(ARRAY_LENGTH)
]

assert input_readback == SORT_INPUT, (
    "DMEM input readback failed"
)

assert read_word(dmem, DONE_OFFSET) == 0, (
    "DONE flag was not reset"
)


DMEM_INPUT_HEX_WORDS = [
    read_word(
        dmem,
        word_index * 4,
    )
    for word_index in range(64)
]

save_hex_file(
    "data.mem",
    DMEM_INPUT_HEX_WORDS,
)

save_hex_file(
    "sort32_dmem_input.hex",
    DMEM_INPUT_HEX_WORDS,
)

print("Instruction Hex created: test_sort.hex")
print("Python loaded the program from test_sort.hex")
print("Random signed input written to DMEM")
print("Input address range: 0x00 through 0x7C")
print("DONE reset to zero at address 0x80")
print("Simulation data file created: data.mem")
print("CPU is held in reset")

Cleared 256 bytes of data memory from 0x00000000
Loaded 64 instructions
Program readback verified
Instruction Hex created: test_sort.hex
Python loaded the program from test_sort.hex
Random signed input written to DMEM
Input address range: 0x00 through 0x7C
DONE reset to zero at address 0x80
Simulation data file created: data.mem
CPU is held in reset


In [15]:
hold_cpu_in_reset()

write_word(
    dmem,
    DONE_OFFSET,
    0,
)

gpio.write(
    GPIO_DATA,
    RUN_RESET_RELEASED,
)

time.sleep(0.05)

gpio.write(
    GPIO_DATA,
    RUN_RESET_ASSERTED,
)

time.sleep(0.01)

diagnostic_done = read_word(
    dmem,
    DONE_OFFSET,
)

print(
    f"Diagnostic DONE value: "
    f"0x{diagnostic_done:08X}"
)

assert diagnostic_done == DONE_MAGIC, (
    f"Expected 0x{DONE_MAGIC:08X}, "
    f"got 0x{diagnostic_done:08X}"
)

print("Sorting program reached DONE")
print("CPU is held in reset")

Diagnostic DONE value: 0xCAFEBABE
Sorting program reached DONE
CPU is held in reset


In [16]:
def run_sort_with_timeout(
    timeout_seconds=1.0,
    initial_delay_seconds=0.02,
):
    assert timeout_seconds > 0, (
        "Timeout must be positive"
    )

    assert initial_delay_seconds >= 0, (
        "Initial delay must not be negative"
    )

    hold_cpu_in_reset()

    write_word(
        dmem,
        DONE_OFFSET,
        0,
    )

    start_time = time.monotonic()

    gpio.write(
        GPIO_DATA,
        RUN_RESET_RELEASED,
    )

    try:
        time.sleep(initial_delay_seconds)

        while True:
            elapsed = (
                time.monotonic()
                - start_time
            )

            if elapsed > timeout_seconds:
                raise TimeoutError(
                    "Sorting program did not finish "
                    "before the timeout"
                )

            done_value = read_word(
                dmem,
                DONE_OFFSET,
            )

            if done_value == DONE_MAGIC:
                return elapsed

            if done_value != 0:
                raise RuntimeError(
                    f"Unexpected status flag: "
                    f"0x{done_value:08X}"
                )

            time.sleep(0.005)

    finally:
        gpio.write(
            GPIO_DATA,
            RUN_RESET_ASSERTED,
        )

        time.sleep(0.01)


elapsed_seconds = run_sort_with_timeout(
    timeout_seconds=1.0,
    initial_delay_seconds=0.02,
)

sorted_readback = [
    signed_word(
        read_word(
            dmem,
            ARRAY_BASE + index * 4,
        )
    )
    for index in range(ARRAY_LENGTH)
]

expected_sorted = sorted(SORT_INPUT)

right_guard_readback = read_word(
    dmem,
    RIGHT_GUARD_OFFSET,
)

done_readback = read_word(
    dmem,
    DONE_OFFSET,
)


print(
    f"Completion time: "
    f"{elapsed_seconds:.6f} seconds"
)

print(
    f"Done flag: "
    f"0x{done_readback:08X}"
)

print("")
print("Random input values:")
print(SORT_INPUT)

print("")
print("Hardware output:")
print(sorted_readback)

print("")
print("Expected output:")
print(expected_sorted)


assert done_readback == DONE_MAGIC, (
    f"Expected status 0x{DONE_MAGIC:08X}, "
    f"got 0x{done_readback:08X}"
)

assert len(sorted_readback) == 32

assert sorted_readback == expected_sorted, (
    "The 32 signed integers were not sorted correctly"
)

assert right_guard_readback == RIGHT_GUARD_VALUE, (
    "Memory after the array was modified"
)

assert ARRAY_BASE == 0x00

assert ARRAY_END - 4 == 0x7C


print("")
print("Exactly 32 random signed integers were sorted")
print("Positive and negative values were included")
print("The result is stored in the original address range")
print("Array range verified: 0x00 through 0x7C")
print("DONE magic verified at address 0x80")
print("Memory guard word is unchanged")
print("Random signed sorting hardware test passed")
print("CPU is held in reset")

Completion time: 0.020398 seconds
Done flag: 0xCAFEBABE

Random input values:
[-186330981, 249585318, 894641537, -624010408, -1003773380, -1960796514, 249913527, -578235842, 608433994, 100786595, 1633279752, 677325013, -362282644, 1741598169, -728255358, -330939495, 1560925635, 1550625964, -2039872380, -534524188, 587641507, 21737741, 1716811657, 470093155, -975196147, -1426982517, 1152064071, -1989937412, -948628915, -159491399, 1872679488, -1429315569]

Hardware output:
[-2039872380, -1989937412, -1960796514, -1429315569, -1426982517, -1003773380, -975196147, -948628915, -728255358, -624010408, -578235842, -534524188, -362282644, -330939495, -186330981, -159491399, 21737741, 100786595, 249585318, 249913527, 470093155, 587641507, 608433994, 677325013, 894641537, 1152064071, 1550625964, 1560925635, 1633279752, 1716811657, 1741598169, 1872679488]

Expected output:
[-2039872380, -1989937412, -1960796514, -1429315569, -1426982517, -1003773380, -975196147, -948628915, -728255358, -62401040

In [17]:
DMEM_OUTPUT_HEX_WORDS = [
    read_word(dmem, word_index * 4)
    for word_index in range(64)
]

save_hex_file(
    "sort32_dmem_output.hex",
    DMEM_OUTPUT_HEX_WORDS,
)

output_file = Path(
    "sort32_dmem_output.hex"
)

assert output_file.is_file(), (
    "Output Hex file was not created"
)

print("Created sort32_dmem_output.hex")
print(f"Output file: {output_file.resolve()}")
print("Sorting test and Hex export completed")

Created sort32_dmem_output.hex
Output file: /home/xilinx/jupyter_notebooks/overlay_pynq/sort32_dmem_output.hex
Sorting test and Hex export completed


In [19]:
from pathlib import Path

hold_cpu_in_reset()

required_files = [
    "sort32_signed.S",
    "sort32_imem.hex",
    "sort32_dmem_input.hex",
    "sort32_dmem_output.hex",
]

assert final_result == 12, (
    "Basic hardware test did not pass"
)

assert full_test_result == 0x12345048, (
    "Full hardware regression test did not pass"
)

assert slt_negative_less_positive == 1
assert slt_positive_less_negative == 0

assert len(sorted_readback) == 32, (
    "Sorting output does not contain exactly 32 integers"
)

assert sorted_readback == expected_sorted, (
    "Signed integer sorting test did not pass"
)

assert done_readback == DONE_MAGIC, (
    "Sorting completion magic value was not written"
)

# assert left_guard_readback == LEFT_GUARD_VALUE, (
#     "Left memory guard was modified"
# )

assert right_guard_readback == RIGHT_GUARD_VALUE, (
    "Right memory guard was modified"
)

for filename in required_files:
    assert Path(filename).is_file(), (
        f"Missing required file: {filename}"
    )

print("Final verification summary")
print(f"IMEM controller: {IMEM_NAME}")
print(f"DMEM controller: {DMEM_NAME}")
print(f"Basic test result: {final_result}")
print(f"Full test result: 0x{full_test_result:08X}")
print("Signed SLT test: passed")
print(f"Sorted integer count: {len(sorted_readback)}")
print("Sorting address range: 0x40 through 0xBC")
print("Signed sorting result: passed")
print(f"Done magic: 0x{done_readback:08X}")
print("Memory guard check: passed")
print("Hex file generation: passed")
print("CPU state: reset")
print("FPGA hardware verification completed successfully")

Final verification summary
IMEM controller: axi_bram_ctrl_0
DMEM controller: axi_bram_ctrl_1
Basic test result: 12
Full test result: 0x12345048
Signed SLT test: passed
Sorted integer count: 32
Sorting address range: 0x40 through 0xBC
Signed sorting result: passed
Done magic: 0xCAFEBABE
Memory guard check: passed
Hex file generation: passed
CPU state: reset
FPGA hardware verification completed successfully
